# OCDD run8: ConvNeXt-Tiny (OPTIONAL second model)
Same data as run7. Compare it with run7 on field-photo accuracy. About 1.5 to 2 hours on a P100.

**Before running:** right side > Session options > **Accelerator = GPU (P100 or T4)**, Persistence = Files only; **Add Input** = your `ocdd-data` dataset.
**Best way to run:** click **Save Version > Save & Run All (Commit)**. It then keeps running on Kaggle's servers even if you close the browser or shut your laptop (9 hour limit). When it finishes, open the notebook's **Output** tab and download the `run8_out` folder.


In [ ]:
import os, glob, shutil, subprocess, sys
inp = "/kaggle/input"
manifest = None; cache = None
for d, dirs, files in os.walk(inp):
    if manifest is None and "manifest_v4.csv" in files: manifest = os.path.join(d, "manifest_v4.csv")
    if cache is None and sum(f.endswith(".jpg") for f in files[:20]) >= 10: cache = d
    if manifest and cache: break
assert manifest and cache, "dataset not found: use Add Input and pick your OCDD dataset"
code_dir = os.path.dirname(manifest)
print("manifest:", manifest); print("cache   :", cache, "(%d files)" % len(os.listdir(cache)))
ROOT = "/kaggle/working/ocdd"
os.makedirs(ROOT + "/data/interim_v4", exist_ok=True); os.makedirs(ROOT + "/data/interim", exist_ok=True)
shutil.copy(manifest, ROOT + "/data/interim_v4/manifest.csv")
if not os.path.exists(ROOT + "/data/interim/cache256"): os.symlink(cache, ROOT + "/data/interim/cache256")
for f in ("train.py", "ctl.py", "progress.py"): shutil.copy(os.path.join(code_dir, f), ROOT)
os.environ["OCDD_ROOT"] = ROOT
import torch; print("torch", torch.__version__, "cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU - turn the accelerator on")
# fail early with a clear message if Internet is off (the ImageNet starting weights are downloaded once)
import urllib.request
try:
    urllib.request.urlopen("https://download.pytorch.org", timeout=10)
    print("internet OK")
except Exception as e:
    raise SystemExit("INTERNET IS OFF: open Session options on the right and switch Internet to On, then run again.")


In [ ]:
# run8 (convnext_tiny). Data and split are identical to the other model, so the two can be compared fairly.
cmd = [sys.executable, "-u", "train.py", "--root", ROOT, "--arch", "convnext_tiny", "--interim", "interim_v4", "--epochs", "15",
       "--batch", "64", "--lr", "2e-4", "--sampling-power", "0.75", "--workers", "4", "--out", "run8"]
p = subprocess.Popen(cmd, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=dict(os.environ, PYTHONUNBUFFERED="1"))
for line in p.stdout:
    print(line, end="")
p.wait(); print("exit code", p.returncode)


In [ ]:
out = "/kaggle/working/run8_out"; os.makedirs(out, exist_ok=True)
for f in ("best.pt", "classes.json", "temperature.json", "report.txt"):
    s = os.path.join(ROOT, "training", "run8", f)
    if os.path.exists(s): shutil.copy(s, out)
print(os.listdir(out)); print(open(out + "/report.txt").read()[:1500])
